# Phase 3: ECRformer-Light Interactive Architecture & Training Walkthrough
### SPARRSO Fellowship: Single-Date SAR-Optical Cloud Removal
**Reference Paper:** *"ECRformer: An efficient cloud removal Transformer with semantic-decoupled learning for multimodal satellite imagery"* (Zhang et al., ISPRS J. Photogramm. Remote Sens., 2026)

---
### Overview
This notebook provides a complete, interactive, cell-by-cell walkthrough of **ECRformer-Light**:
1. **XCA (Cross-Covariance Attention):** Shifting attention from spatial tokens $N$ to channels $C$ with $\mathcal{O}(NC^2)$ complexity.
2. **MDWA (Multi-Dilation Window Attention):** Multi-scale spatial receptive fields with parallel dilation rates $d \in [1, 2, 3, 4]$ and linear $\mathcal{O}(N)$ complexity.
3. **SGFN (Self-Gated Feed-Forward Network):** $3\times 3$ Depthwise Conv with GELU element-wise gating.
4. **ECRformer Block:** Serial integration of XCA $\rightarrow$ MDWA $\rightarrow$ SGFN.
5. **Shallow Feature Embedding:** Modality-specific parallel $7\times 7$ convolutions for Optical and SAR.
6. **Full U-Shaped Backbone:** 3 Encoder stages, Bottleneck, 3 Decoder stages with skip connections, and Refinement network (Total: 1.56M params).
7. **SDFL Training Objective:** Encoder Structural SSIM loss + Decoder Textural L1 loss.
8. **Real Patch End-to-End Test:** Forward pass, loss computation, and gradient backpropagation on a real satellite patch.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[✓] PyTorch Version: {torch.__version__} | Active Device: {device}")

## 1. Cross-Covariance Attention (XCA) - Section 3.3.1 (Eq. 10 & 11)
### Mathematical Core:
Standard self-attention computes an $N \times N$ attention matrix ($N = H \times W$), requiring $\mathcal{O}(N^2 C)$ operations, which is prohibitive for satellite imagery.
**XCA transposes the attention to channels** ($C \times C$ covariance matrix):
$$\text{XCAttention}(Q, K, V) = V \cdot \text{Softmax}\left( \frac{K^T Q}{\tau} \right)$$
- $\tau$ is a learnable per-head temperature parameter scaling the logits.
- Complexity: $\mathcal{O}(N C^2)$ — strictly linear in spatial resolution $N$!
- Cross-modal role: Learns correlations between SAR backscatter channels and optical reflectance bands.

In [ ]:
class CrossCovarianceAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int = 4):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        
        # 1x1 conv followed by 3x3 depthwise conv to capture local context (Eq. 11)
        self.qkv = nn.Sequential(
            nn.Conv2d(dim, dim * 3, kernel_size=1, bias=False),
            nn.Conv2d(dim * 3, dim * 3, kernel_size=3, padding=1, groups=dim * 3, bias=False)
        )
        self.project_out = nn.Conv2d(dim, dim, kernel_size=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        N = H * W
        qkv = self.qkv(x)
        q, k, v = torch.chunk(qkv, 3, dim=1)
        
        # Reshape to [B, heads, head_dim, N]
        q = q.reshape(B, self.num_heads, self.head_dim, N)
        k = k.reshape(B, self.num_heads, self.head_dim, N)
        v = v.reshape(B, self.num_heads, self.head_dim, N)
        
        # L2-normalize queries and keys along spatial dimension N
        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)
        
        # Cross-covariance matrix: [head_dim x head_dim]
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.temperature
        attn = F.softmax(attn, dim=-1)
        
        # Aggregate values: [B, heads, head_dim, N]
        out = torch.matmul(attn, v).reshape(B, C, H, W)
        return self.project_out(out)

# Test XCA
xca = CrossCovarianceAttention(dim=32, num_heads=4).to(device)
dummy_in = torch.randn(2, 32, 64, 64, device=device)
out_xca = xca(dummy_in)
print(f"[✓] XCA Input: {dummy_in.shape} -> Output: {out_xca.shape}")

## 2. Multi-Dilation Window Attention (MDWA) - Section 3.3.2 (Eq. 12 - 14)
### Mathematical Core:
To capture multi-scale spatial context (object boundaries, road grids, and cloud margins):
1. **Conditional Positional Encoding (CPE):** $3\times 3$ Depthwise Conv injected into features before projection (Eq. 12).
2. **Dilated Sampling:** For window size $M=3$, parallel branches sample at dilation rates $d \in [1, 2, 3, 4]$ (receptive fields $3\times 3$, $5\times 5$, $7\times 7$, $9\times 9$).
3. **Attention:** Each central query attends to its 9 dilated neighbors:
$$\text{DilatedAttn}(h, w) = \text{Softmax}\left( \frac{q(h, w) K_{\text{dilated}}^T}{\sqrt{d_k}} \right) V_{\text{dilated}}$$
4. **Fusion:** Parallel branch outputs are fused via a $1\times 1$ convolution.

In [ ]:
class MultiDilationWindowAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int = 4, window_size: int = 3, dilation_rates=[1, 2, 3, 4]):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.window_size = window_size
        self.dilation_rates = dilation_rates
        
        # Conditional Positional Encoding (CPE): 3x3 DWConv (Eq. 12)
        self.cpe = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim, bias=False)
        self.qkv = nn.Conv2d(dim, dim * 3, kernel_size=1, bias=False)
        self.conv_fuse = nn.Conv2d(dim * len(dilation_rates), dim, kernel_size=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        N = H * W
        x_cpe = x + self.cpe(x)
        qkv = self.qkv(x_cpe)
        q, k, v = torch.chunk(qkv, 3, dim=1)
        q_view = q.view(B, self.num_heads, self.head_dim, 1, N)
        
        branch_outputs = []
        for d in self.dilation_rates:
            # Extract 3x3 dilated spatial neighborhood
            k_u = F.unfold(k, kernel_size=self.window_size, dilation=d, padding=d).view(B, self.num_heads, self.head_dim, 9, N)
            v_u = F.unfold(v, kernel_size=self.window_size, dilation=d, padding=d).view(B, self.num_heads, self.head_dim, 9, N)
            attn = F.softmax((q_view * k_u).sum(dim=2) * self.scale, dim=2)
            out_d = (attn.unsqueeze(2) * v_u).sum(dim=3).view(B, C, H, W)
            branch_outputs.append(out_d)
            
        return self.conv_fuse(torch.cat(branch_outputs, dim=1))

# Test MDWA
mdwa = MultiDilationWindowAttention(dim=32, num_heads=4, dilation_rates=[1, 2, 3, 4]).to(device)
out_mdwa = mdwa(dummy_in)
print(f"[✓] MDWA Input: {dummy_in.shape} -> Output: {out_mdwa.shape}")

## 3. Self-Gated Feed-forward Network (SGFN) - Section 3.3.3 (Eq. 15 & 16)
### Mathematical Core:
Replaces standard MLP with depthwise convolution and GELU gating:
$$X_1, X_2 = \text{Split}\left( \text{DWConv}_{3\times 3}(\text{Conv}_{1\times 1}(F_{\text{in}})) \right)$$
$$F_{\text{out}} = \text{Conv}_{1\times 1}\left( \text{GELU}(X_1) \odot X_2 \right)$$
- Hidden expansion factor = $2.0$ (Table 6).
- Enhances non-linear expressive power while preserving local spatial inductive bias.

In [ ]:
class SelfGatedFeedForward(nn.Module):
    def __init__(self, dim: int, expansion_factor: float = 2.0):
        super().__init__()
        hidden_dim = int(dim * expansion_factor)
        self.project_in = nn.Conv2d(dim, hidden_dim * 2, kernel_size=1, bias=False)
        self.dwconv = nn.Conv2d(hidden_dim * 2, hidden_dim * 2, kernel_size=3, padding=1, groups=hidden_dim * 2, bias=False)
        self.project_out = nn.Conv2d(hidden_dim, dim, kernel_size=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_proj = self.project_in(x)
        x_dw = self.dwconv(x_proj)
        x1, x2 = torch.chunk(x_dw, 2, dim=1)
        return self.project_out(F.gelu(x1) * x2)

# Test SGFN
sgfn = SelfGatedFeedForward(dim=32, expansion_factor=2.0).to(device)
out_sgfn = sgfn(dummy_in)
print(f"[✓] SGFN Input: {dummy_in.shape} -> Output: {out_sgfn.shape}")

## 4. The Complete ECRformer Block - Section 3.3 (Eq. 7 - 9)
Serially connects the three specialized components with 2D LayerNorm:
$$F_{\text{XCA}}  = \text{XCA}(\text{LN}(F_{\text{in}})) + F_{\text{in}} \quad \text{(Eq. 7)}$$
$$F_{\text{MDWA}} = \text{MDWA}(\text{LN}(F_{\text{XCA}})) + F_{\text{XCA}} \quad \text{(Eq. 8)}$$
$$F_{\text{out}}  = \text{SGFN}(\text{LN}(F_{\text{MDWA}})) + F_{\text{MDWA}} \quad \text{(Eq. 9)}$$

In [ ]:
class LayerNorm2d(nn.Module):
    def __init__(self, channels: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(channels))
        self.bias = nn.Parameter(torch.zeros(channels))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=1, keepdim=True)
        var = (x - mean).pow(2).mean(dim=1, keepdim=True)
        return self.weight[:, None, None] * ((x - mean) / torch.sqrt(var + self.eps)) + self.bias[:, None, None]

class ECRformerBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int = 4, dilation_rates=[1, 2, 3, 4], expansion_factor=2.0):
        super().__init__()
        self.norm1 = LayerNorm2d(dim)
        self.xca = CrossCovarianceAttention(dim, num_heads=num_heads)
        self.norm2 = LayerNorm2d(dim)
        self.mdwa = MultiDilationWindowAttention(dim, num_heads=num_heads, dilation_rates=dilation_rates)
        self.norm3 = LayerNorm2d(dim)
        self.sgfn = SelfGatedFeedForward(dim, expansion_factor=expansion_factor)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.xca(self.norm1(x))
        x = x + self.mdwa(self.norm2(x))
        x = x + self.sgfn(self.norm3(x))
        return x

# Test ECRformerBlock
block = ECRformerBlock(dim=32).to(device)
out_block = block(dummy_in)
print(f"[✓] ECRformer Block Input: {dummy_in.shape} -> Output: {out_block.shape}")

## 5. Full U-Shaped ECRformer-Light Backbone (Table 6 Specifications)
- **Embedding Dimension:** $C_{\text{embed}} = 32$
- **Channel Dims across Stages:** $[32, 64, 128]$
- **Encoder Stages:** $K_{\text{enc}} = 3$, Depths $L = [2, 2, 1]$
- **Bottleneck:** $L_B = 1$ (dim: 128)
- **Decoder Stages:** $K_{\text{dec}} = 3$, Depths $L = [1, 2, 2]$ with Skip Connections
- **Refinement Network:** $L_R = 2$ full-resolution blocks + $3\times 3$ Conv
- **SDFL Projection Heads:** Multi-scale deep supervision heads on intermediate stages

In [ ]:
from models.ecrformer_light import ECRformerLight, SDFLLoss

# Instantiate the official paper-accurate model
model = ECRformerLight(
    in_channels_opt=4,   # B2, B3, B4, B8
    in_channels_sar=2,   # VV, VH
    out_channels=4,
    embed_dim=32,
    encoder_depths=[2, 2, 1],
    bottleneck_depth=1,
    decoder_depths=[1, 2, 2],
    refinement_depth=2,
    num_heads=4,
    dilation_rates=[1, 2, 3, 4]
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[✓] ECRformer-Light Instantiated Successfully!")
print(f"    Total Trainable Parameters: {n_params:,} ({n_params / 1e6:.2f}M params)")
print(f"    (Paper Base Model: 11.29M | Light Variant: 1.56M on 4-band Optical + 2-band SAR)")

## 6. Semantic-Decoupled Feature Learning (SDFL Loss) - Section 3.4
### Probabilistic Factorization:
$$p(Y | X_{\text{opt}}, X_{\text{sar}}) = \int p(Y | S) \cdot p(S | X_{\text{opt}}, X_{\text{sar}}) \, dS$$
1. **Encoder Learns Scene Structure ($S$):** Enforced via **SSIM Structural Loss**:
$$\mathcal{R}_{\text{enc}} = \mathbb{E}_k [1 - \text{SSIM}(\hat{Y}_{\text{enc}}^{(k)}, Y^{(k)})]$$
2. **Decoder Renders Surface Texture ($Y$):** Enforced via **L1 Textural Loss**:
$$\mathcal{R}_{\text{dec}} = \mathbb{E}_k [\|\hat{Y}_{\text{dec}}^{(k)} - Y^{(k)}\|_1]$$
3. **Combined Training Objective (Eq. 23):**
$$\mathcal{L} = 0.9 \cdot L_1 + 0.1 \cdot (1 - \text{SSIM}) + 0.05 \cdot \mathcal{R}_{\text{enc}} + 0.05 \cdot \mathcal{R}_{\text{dec}}$$

In [ ]:
# Test loss on synthetic tensors
criterion = SDFLLoss(alpha=0.9, beta=0.1, lambda_enc=0.05, lambda_dec=0.05)

dummy_opt = torch.randn(2, 4, 128, 128, device=device)
dummy_sar = torch.randn(2, 2, 128, 128, device=device)
dummy_gt  = torch.randn(2, 4, 128, 128, device=device)

# Forward pass with multi-scale deep supervision heads
preds = model(dummy_opt, dummy_sar, return_sdfl=True)
losses = criterion(preds, dummy_gt)

print("[✓] SDFL Loss Terms:")
print(f"    Total Objective Loss: {losses['loss'].item():.4f}")
print(f"    Output L1 Loss:       {losses['loss_l1'].item():.4f}")
print(f"    Output SSIM Loss:     {losses['loss_ssim'].item():.4f}")
print(f"    Encoder Struct SSIM:  {losses['loss_r_enc'].item():.4f}")
print(f"    Decoder Texture L1:   {losses['loss_r_dec'].item():.4f}")

## 7. Real Satellite Patch End-to-End Test & Training Step
We load a real extracted patch from our dataset (`data/patches_256/train/`), perform a forward pass through ECRformer-Light, compute the SDFL loss, execute backpropagation, and visualize the output.

In [ ]:
import glob
import os

# Locate real patch from Phase 2 preprocessing
patch_files = glob.glob("./data/patches_256/train/*.npz") or glob.glob("../data/patches_256/train/*.npz")
assert len(patch_files) > 0, "No patches found! Run Phase 2 preprocessing first."

patch_data = np.load(patch_files[0])
real_opt = torch.from_numpy(patch_data['optical']).unsqueeze(0).to(device)  # [1, 4, 256, 256]
real_sar = torch.from_numpy(patch_data['sar']).unsqueeze(0).to(device)      # [1, 2, 256, 256]
real_target = real_opt.clone()  # Target

print(f"[*] Real Patch Loaded: {patch_files[0]}")
print(f"    Optical: {real_opt.shape} | SAR: {real_sar.shape}")

# Training Step Simulation
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=1e-2)
optimizer.zero_grad()

preds = model(real_opt, real_sar, return_sdfl=True)
loss_dict = criterion(preds, real_target)
loss_dict['loss'].backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

print(f"[✓] Training Step Succeeded! Loss = {loss_dict['loss'].item():.4f}")

# Visualize Reconstruction Comparison
with torch.no_grad():
    model.eval()
    clean_pred = model(real_opt, real_sar, return_sdfl=False)[0].cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Input Cloudy RGB
in_rgb = np.stack([real_opt[0, 2].cpu(), real_opt[0, 1].cpu(), real_opt[0, 0].cpu()], axis=-1)
axes[0].imshow(np.clip(in_rgb / 0.35, 0, 1))
axes[0].set_title("(a) Input Cloudy Optical (RGB)", fontweight='bold')
axes[0].axis('off')

# 2. Co-registered SAR VV
axes[1].imshow(real_sar[0, 0].cpu(), cmap='gray', vmin=-1, vmax=1)
axes[1].set_title("(b) Input Sentinel-1 SAR VV (dB)", fontweight='bold')
axes[1].axis('off')

# 3. ECRformer-Light Output
out_rgb = np.stack([clean_pred[2], clean_pred[1], clean_pred[0]], axis=-1)
axes[2].imshow(np.clip(out_rgb / 0.35, 0, 1))
axes[2].set_title("(c) ECRformer-Light Output (Initial Step)", fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Summary of ECRformer-Light Implementation

| Component | Paper Reference | Implementation Detail |
| :--- | :--- | :--- |
| **Modality Fusion** | Section 3.2.1, Eq. 2 | Parallel $7\times 7$ convolutions on Optical and SAR, then concatenated |
| **Channel Attention** | Section 3.3.1, Eq. 10 | Cross-Covariance Attention (XCA), $\mathcal{O}(NC^2)$ complexity with learnable $\tau$ |
| **Spatial Attention** | Section 3.3.2, Eq. 12-14 | Multi-Dilation Window Attention (MDWA), $M=3, d \in [1, 2, 3, 4]$, CPE $3\times 3$ DWConv |
| **Feed-Forward Net** | Section 3.3.3, Eq. 15-16 | Self-Gated Feed-forward Network (SGFN) with expansion $2.0$ & GELU gating |
| **Lightweight Scale** | Table 6 | 1.56M parameters (vs. 11.29M base model) |
| **Training Objective** | Section 3.4, Eq. 23 | Semantic-Decoupled Feature Learning (SDFL): Encoder SSIM + Decoder L1 |